# 🔬 Galvanic Corrosion PINN — Cloud Training Notebook

**Physics-Informed Graph Neural Network for Galvanic Corrosion Prediction**

This notebook trains a Physics-Informed GNN (PINN-GNN) to predict:
1. **Galvanic current density** (regression) — corrosion rate proxy
2. **Material compatibility** (classification) — Compatible vs Unfavorable

Based on AMMTIAC corrosion data with NNConv message passing and physics-informed loss (KCL + thermodynamic constraints).

---

**Instructions:**
1. Set runtime to **GPU** (Runtime → Change runtime type → T4 GPU)
2. Upload all project `.py` and `.csv` files to `/content/corrosion/`
3. Run all cells sequentially

> ⚡ Estimated training time: ~5-10 min on T4 GPU (200 epochs)

## 📦 Step 0: Install Dependencies

In [ ]:
%%time
import subprocess, sys, os

# ── Project directory (all paths are absolute from here) ──
PROJECT_DIR = "/content/corrosion"
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)

# Install torch-geometric (PyTorch is pre-installed on Colab)
try:
    import torch_geometric
    print(f"✓ PyG already installed: {torch_geometric.__version__}")
except ImportError:
    print("Installing torch-geometric...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "torch-geometric", "-q"])
    import torch_geometric
    print(f"✓ PyG installed: {torch_geometric.__version__}")

# Verify environment
import torch
print(f"✓ PyTorch: {torch.__version__}")
print(f"✓ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✓ GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
print(f"\n✓ All dependencies ready.")
print(f"✓ Working directory: {os.getcwd()}")

## 📁 Step 1: Setup Project Files

Upload your project files or clone from your repo.

In [ ]:
# ── Path helper (all cells use this for absolute paths) ──
def P(relative_path):
    """Resolve a path relative to PROJECT_DIR. Always returns absolute path."""
    return os.path.join(PROJECT_DIR, relative_path)

# Ensure we're in the right directory
os.chdir(PROJECT_DIR)

# Check if files are already present
required_files = [
    "data_extraction.py",
    "graph_dataset.py",
    "pinn_model.py",
    "train.py",
]

missing = [f for f in required_files if not os.path.exists(P(f))]
if missing:
    print("⚠ Missing files — please upload them:")
    for f in missing:
        print(f"   • {f}")
    print(f"\n📤 Upload to: {PROJECT_DIR}")
    print('   Or run: from google.colab import files; files.upload()')
else:
    print(f"✓ All required files found in {PROJECT_DIR}")
    print(f"  Files: {os.listdir(PROJECT_DIR)}")

In [ ]:
# ── Upload files interactively (uncomment if needed) ──
# from google.colab import files
# uploaded = files.upload()
# print(f"\n✓ Uploaded {len(uploaded)} files")

## 🧪 Step 2: Generate Synthetic Training Data

Extracts reference tables from AMMTIAC data and generates 25,000 synthetic galvanic joint records with physics-based targets.

In [ ]:
%%time
os.chdir(PROJECT_DIR)

from data_extraction import (
    extract_emf_series,
    extract_material_groups,
    extract_galvanic_series,
    generate_synthetic_data,
)

print("Extracting reference tables...")
df_emf = extract_emf_series()
df_groups = extract_material_groups()
df_galvanic = extract_galvanic_series()

print("\nGenerating synthetic dataset...")
NUM_SAMPLES = 25000
df_synth = generate_synthetic_data(df_galvanic, df_groups, num_samples=NUM_SAMPLES)

print(f"\n{'='*60}")
print(f"  ✓ {len(df_synth)} synthetic joints generated")
print(f"  ✓ {len(df_galvanic)} materials in galvanic series")
print(f"  ✓ {len(df_groups)} compatibility groups")
print(f"  ✓ {len(df_emf)} EMF equilibria")
print(f"{'='*60}")

In [ ]:
# Preview the generated data
print("\n── Synthetic Dataset Preview ──")
print(f"Columns: {list(df_synth.columns)}")
print(f"Shape: {df_synth.shape}")
display(df_synth.head(10))

print("\n── Distribution Stats ──")
print(f"\nEnvironment distribution:")
print(df_synth["Environment"].value_counts().to_string())
print(f"\nCompatibility status:")
print(df_synth["PDF_Compatibility_Status"].value_counts().to_string())
print(f"\nTarget current density: min={df_synth['Target_Galvanic_Current_Density'].min():.6f}, "
      f"max={df_synth['Target_Galvanic_Current_Density'].max():.4f}, "
      f"mean={df_synth['Target_Galvanic_Current_Density'].mean():.6f}")

## 📊 Step 3: Data Exploration & Visualization

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Galvanic Corrosion Dataset — Exploratory Analysis", fontsize=14, fontweight='bold')

axes[0, 0].hist(df_synth["Target_Galvanic_Current_Density"], bins=100, color="steelblue", edgecolor="white", alpha=0.8)
axes[0, 0].set_xlabel("Galvanic Current Density (A/m²)")
axes[0, 0].set_ylabel("Count")
axes[0, 0].set_title("Current Density Distribution")

log_current = np.log10(df_synth["Target_Galvanic_Current_Density"].clip(lower=1e-8))
axes[0, 1].hist(log_current, bins=80, color="darkorange", edgecolor="white", alpha=0.8)
axes[0, 1].set_xlabel("log10(Current Density)")
axes[0, 1].set_ylabel("Count")
axes[0, 1].set_title("Current Density (Log Scale)")

colors = df_synth["PDF_Compatibility_Status"].map({"Compatible": "green", "Unfavorable": "red"})
axes[0, 2].scatter(df_synth["Potential_Difference_V"], df_synth["Target_Galvanic_Current_Density"],
                   c=colors, alpha=0.15, s=5)
axes[0, 2].set_xlabel("Potential Difference (V)")
axes[0, 2].set_ylabel("Current Density (A/m2)")
axes[0, 2].set_title("dV vs Current (green=Compatible, red=Unfavorable)")

axes[1, 0].hist(np.log10(df_synth["Cathode_Anode_Area_Ratio"]), bins=80, color="purple", edgecolor="white", alpha=0.8)
axes[1, 0].set_xlabel("log10(Cathode/Anode Area Ratio)")
axes[1, 0].set_ylabel("Count")
axes[1, 0].set_title("Area Ratio Distribution")

env_counts = df_synth["Environment"].value_counts()
axes[1, 1].barh(env_counts.index, env_counts.values, color=["#2196F3", "#FF9800", "#4CAF50"])
axes[1, 1].set_xlabel("Count")
axes[1, 1].set_title("Environment Distribution")

axes[1, 2].barh(df_galvanic["Material"], df_galvanic["Potential_V_SCE"],
                color=plt.cm.RdYlGn(np.linspace(0, 1, len(df_galvanic))))
axes[1, 2].set_xlabel("Potential (V vs SCE)")
axes[1, 2].set_title("Galvanic Series")
axes[1, 2].tick_params(axis='y', labelsize=6)

plt.tight_layout()
plt.savefig(P("data_exploration.png"), dpi=150, bbox_inches='tight')
plt.show()
print("Saved data_exploration.png")

## 🔗 Step 4: Build Graph Dataset

Converts each galvanic joint into a 2-node PyG graph:
- **Nodes**: anode & cathode with 22-dim features (SCE potential, rank, group one-hot)
- **Edges**: bidirectional with 6-dim features (dV, area ratio, conductivity, environment)
- **Targets**: current density (regression) + compatibility (classification)

In [ ]:
%%time
from graph_dataset import build_graph_list

SYNTH_CSV = P("synthetic_galvanic_joints_full.csv")
GALV_CSV = P("pdf_table2_galvanic_series.csv")

print("Converting CSV to PyG graph objects...")
train_graphs, val_graphs, test_graphs = build_graph_list(
    SYNTH_CSV, GALV_CSV, split="train", seed=42
)

print(f"\n  Train: {len(train_graphs):,} graphs")
print(f"  Val:   {len(val_graphs):,} graphs")
print(f"  Test:  {len(test_graphs):,} graphs")

g = train_graphs[0]
print(f"\n  Sample graph structure:")
print(f"    Nodes: {g.x.shape}  (2 nodes x {g.x.shape[1]} features)")
print(f"    Edges: {g.edge_index.shape}")
print(f"    Edge attr: {g.edge_attr.shape}  ({g.edge_attr.shape[1]} features)")
print(f"    y_regression:     {g.y_regression.item():.6f}")
print(f"    y_classification: {g.y_classification.item():.0f}")

zero_nodes = sum(1 for g in train_graphs if (g.x.abs().sum(dim=1) == 0).any())
if zero_nodes > 0:
    print(f"\n  WARNING: {zero_nodes} graphs have zero-feature nodes (material lookup failed)")
else:
    print(f"\n  All node features populated correctly (no lookup failures)")

## 🧠 Step 5: Initialize Model

Architecture: **NNConv** (edge-aware message passing) with residual connections, LayerNorm, and 3 prediction heads.

In [ ]:
from pinn_model import GalvanicCorrosionPINN, PhysicsInformedLoss, build_model
from torch_geometric.loader import DataLoader

# Hyperparameters
EPOCHS = 200
BATCH_SIZE = 128
LR = 1e-3
HIDDEN_DIM = 64
NUM_MP_LAYERS = 3
DROPOUT = 0.1
LAMBDA_KCL = 0.1
LAMBDA_THERMO = 0.1
LAMBDA_CLS = 0.5
SEED = 42

# Output paths (absolute)
CHECKPOINT_DIR = P("checkpoints")
BEST_MODEL_PATH = P("checkpoints/best_model.pt")
FINAL_MODEL_PATH = P("checkpoints/final_model.pt")
LOG_PATH = P("training_log.csv")

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
np.random.seed(SEED)
if device.type == "cuda":
    torch.cuda.manual_seed_all(SEED)

# DataLoaders
train_loader = DataLoader(train_graphs, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_graphs, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_graphs, batch_size=BATCH_SIZE, shuffle=False)

# Model
sample = train_graphs[0]
node_features = sample.x.shape[1]
edge_features = sample.edge_attr.shape[1]

model = build_model(
    node_features=node_features,
    edge_features=edge_features,
    hidden_dim=HIDDEN_DIM,
    num_mp_layers=NUM_MP_LAYERS,
    dropout=DROPOUT,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Device: {device}")
print(f"Model: {model.__class__.__name__}")
print(f"  Node features: {node_features}")
print(f"  Edge features: {edge_features}")
print(f"  Hidden dim:    {HIDDEN_DIM}")
print(f"  MP layers:     {NUM_MP_LAYERS}")
print(f"  Parameters:    {total_params:,} ({trainable_params:,} trainable)")

# Optimizer & Scheduler
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

optimizer = Adam(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR * 0.01)

# Loss
criterion = PhysicsInformedLoss(
    lambda_kcl=LAMBDA_KCL,
    lambda_thermo=LAMBDA_THERMO,
    lambda_cls=LAMBDA_CLS,
)

print(f"\n  Checkpoints -> {CHECKPOINT_DIR}")
print(f"  Log        -> {LOG_PATH}")
print(f"\nModel initialized and ready for training.")

## 🏋 Step 6: Training Loop

In [ ]:
import time, csv
import torch.nn.functional as F

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_losses = {"total": 0, "data": 0, "classification": 0, "kcl": 0, "thermo": 0}
    n_batches = 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        v_nodes, i_edges, compat = model(batch)
        i_true = batch.y_regression.unsqueeze(-1) if batch.y_regression.dim() == 1 else batch.y_regression
        num_graphs = batch.num_graphs
        i_pred_per_graph = i_edges[0::2]
        if i_pred_per_graph.shape[0] != i_true.shape[0]:
            i_pred_per_graph = i_edges.view(num_graphs, 2, 1).mean(dim=1)
        compat_true = batch.y_classification.unsqueeze(-1) if batch.y_classification.dim() == 1 else batch.y_classification
        losses = criterion(v_nodes, i_edges, i_pred_per_graph, i_true, compat, compat_true, batch.edge_index)
        losses["total"].backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        for k in total_losses:
            total_losses[k] += losses[k].item()
        n_batches += 1
    for k in total_losses:
        total_losses[k] /= max(n_batches, 1)
    return total_losses


def compute_metrics(model, loader, criterion, device):
    model.eval()
    total_losses = {"total": 0, "data": 0, "classification": 0, "kcl": 0, "thermo": 0}
    all_y_true_reg, all_y_pred_reg = [], []
    all_y_true_cls, all_y_pred_cls = [], []
    n_batches = 0
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            v_nodes, i_edges, compat = model(batch)
            i_true = batch.y_regression.unsqueeze(-1) if batch.y_regression.dim() == 1 else batch.y_regression
            num_graphs = batch.num_graphs
            i_pred_per_graph = i_edges[0::2]
            if i_pred_per_graph.shape[0] != i_true.shape[0]:
                i_pred_per_graph = i_edges.view(num_graphs, 2, 1).mean(dim=1)
            compat_true = batch.y_classification.unsqueeze(-1) if batch.y_classification.dim() == 1 else batch.y_classification
            losses = criterion(v_nodes, i_edges, i_pred_per_graph, i_true, compat, compat_true, batch.edge_index)
            for k in total_losses:
                total_losses[k] += losses[k].item()
            n_batches += 1
            all_y_true_reg.append(i_true.cpu().numpy())
            all_y_pred_reg.append(i_pred_per_graph.cpu().numpy())
            all_y_true_cls.append(compat_true.cpu().numpy())
            all_y_pred_cls.append(torch.sigmoid(compat).cpu().numpy())
    for k in total_losses:
        total_losses[k] /= max(n_batches, 1)
    y_true = np.concatenate(all_y_true_reg)
    y_pred = np.concatenate(all_y_pred_reg)
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    cls_true = np.concatenate(all_y_true_cls).flatten()
    cls_pred = (np.concatenate(all_y_pred_cls).flatten() >= 0.5).astype(float)
    accuracy = np.mean(cls_true == cls_pred)
    tp = np.sum((cls_pred == 1) & (cls_true == 1))
    fp = np.sum((cls_pred == 1) & (cls_true == 0))
    fn = np.sum((cls_pred == 0) & (cls_true == 1))
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-8)
    return {
        "loss_total": total_losses["total"], "loss_data": total_losses["data"],
        "loss_cls": total_losses["classification"], "loss_kcl": total_losses["kcl"],
        "loss_thermo": total_losses["thermo"],
        "mae": mae, "rmse": rmse, "accuracy": accuracy, "f1": f1,
    }

In [ ]:
%%time
# Training
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
best_val_loss = float("inf")

# CSV log - written incrementally so it survives kernel restarts
log_fields = [
    "epoch", "lr",
    "train_total", "train_data", "train_cls", "train_kcl", "train_thermo",
    "val_total", "val_data", "val_cls", "val_kcl", "val_thermo",
    "val_mae", "val_rmse", "val_accuracy", "val_f1",
]
with open(LOG_PATH, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=log_fields)
    writer.writeheader()

print(f"{'='*80}")
print(f"  Training for {EPOCHS} epochs  |  LR: {LR}  |  Batch: {BATCH_SIZE}  |  Hidden: {HIDDEN_DIM}")
print(f"  Device: {device}")
print(f"{'='*80}\n")

start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()

    train_losses = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_metrics = compute_metrics(model, val_loader, criterion, device)

    current_lr = optimizer.param_groups[0]["lr"]
    scheduler.step()

    # Write to CSV log (append each epoch)
    log_row = {
        "epoch": epoch, "lr": current_lr,
        "train_total": train_losses["total"],
        "train_data": train_losses["data"],
        "train_cls": train_losses["classification"],
        "train_kcl": train_losses["kcl"],
        "train_thermo": train_losses["thermo"],
        "val_total": val_metrics["loss_total"],
        "val_data": val_metrics["loss_data"],
        "val_cls": val_metrics["loss_cls"],
        "val_kcl": val_metrics["loss_kcl"],
        "val_thermo": val_metrics["loss_thermo"],
        "val_mae": val_metrics["mae"],
        "val_rmse": val_metrics["rmse"],
        "val_accuracy": val_metrics["accuracy"],
        "val_f1": val_metrics["f1"],
    }
    with open(LOG_PATH, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=log_fields)
        writer.writerow(log_row)

    # Checkpoint
    marker = ""
    if val_metrics["loss_total"] < best_val_loss:
        best_val_loss = val_metrics["loss_total"]
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "val_loss": best_val_loss,
        }, BEST_MODEL_PATH)
        marker = " *"

    epoch_time = time.time() - epoch_start
    if epoch % 10 == 0 or epoch == 1 or marker:
        print(
            f"Epoch {epoch:4d}/{EPOCHS} "
            f"| Train: {train_losses['total']:.5f} "
            f"| Val: {val_metrics['loss_total']:.5f} "
            f"| MAE: {val_metrics['mae']:.5f} "
            f"| Acc: {val_metrics['accuracy']:.3f} "
            f"| F1: {val_metrics['f1']:.3f} "
            f"| LR: {current_lr:.1e} "
            f"| {epoch_time:.1f}s{marker}"
        )

# Save final model
torch.save({
    "epoch": EPOCHS,
    "model_state_dict": model.state_dict(),
}, FINAL_MODEL_PATH)

total_time = time.time() - start_time
print(f"\n{'='*80}")
print(f"  Training complete in {total_time/60:.1f} minutes")
print(f"  Best validation loss: {best_val_loss:.6f}")
print(f"  Checkpoint: {BEST_MODEL_PATH}")
print(f"  Log:        {LOG_PATH}")
print(f"{'='*80}")

## 📈 Step 7: Training Curves & Analysis

In [ ]:
# Load training history from saved CSV (robust to kernel restarts)
df_history = pd.read_csv(P("training_log.csv"))
print(f"Loaded {len(df_history)} epochs from training_log.csv")

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Training Results - Galvanic Corrosion PINN", fontsize=14, fontweight='bold')

axes[0, 0].plot(df_history["epoch"], df_history["train_total"], label="Train", linewidth=1.5)
axes[0, 0].plot(df_history["epoch"], df_history["val_total"], label="Val", linewidth=1.5)
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("Total Loss")
axes[0, 0].set_title("Total Loss (log scale)")
axes[0, 0].set_yscale("log")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(df_history["epoch"], df_history["train_data"], label="Train", linewidth=1.5)
axes[0, 1].plot(df_history["epoch"], df_history["val_data"], label="Val", linewidth=1.5)
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].set_ylabel("MSE Loss")
axes[0, 1].set_title("Regression Loss (MSE)")
axes[0, 1].set_yscale("log")
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

axes[0, 2].plot(df_history["epoch"], df_history["train_cls"], label="Train", linewidth=1.5)
axes[0, 2].plot(df_history["epoch"], df_history["val_cls"], label="Val", linewidth=1.5)
axes[0, 2].set_xlabel("Epoch")
axes[0, 2].set_ylabel("BCE Loss")
axes[0, 2].set_title("Classification Loss (BCE)")
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

axes[1, 0].plot(df_history["epoch"], df_history["val_mae"], label="MAE", color="orange", linewidth=1.5)
axes[1, 0].plot(df_history["epoch"], df_history["val_rmse"], label="RMSE", color="red", linewidth=1.5)
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].set_ylabel("Error (A/m2)")
axes[1, 0].set_title("Validation MAE & RMSE")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(df_history["epoch"], df_history["val_accuracy"], label="Accuracy", color="green", linewidth=1.5)
axes[1, 1].plot(df_history["epoch"], df_history["val_f1"], label="F1 Score", color="blue", linewidth=1.5)
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].set_ylabel("Score")
axes[1, 1].set_title("Classification Metrics")
axes[1, 1].set_ylim([0, 1.05])
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

axes[1, 2].plot(df_history["epoch"], df_history["train_kcl"], label="KCL (train)", linewidth=1.5)
axes[1, 2].plot(df_history["epoch"], df_history["val_kcl"], label="KCL (val)", linewidth=1.5, linestyle='--')
axes[1, 2].plot(df_history["epoch"], df_history["train_thermo"], label="Thermo (train)", linewidth=1.5)
axes[1, 2].plot(df_history["epoch"], df_history["val_thermo"], label="Thermo (val)", linewidth=1.5, linestyle='--')
axes[1, 2].set_xlabel("Epoch")
axes[1, 2].set_ylabel("Penalty")
axes[1, 2].set_title("Physics Constraint Losses")
axes[1, 2].set_yscale("log")
axes[1, 2].legend(fontsize=8)
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(P("training_curves.png"), dpi=150, bbox_inches='tight')
plt.show()
print("Saved training_curves.png")

## 🧪 Step 8: Test Set Evaluation

In [ ]:
# Load best model
print(f"Loading checkpoint from: {BEST_MODEL_PATH}")
ckpt = torch.load(BEST_MODEL_PATH, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
print(f"Loaded best model from epoch {ckpt['epoch']}")

test_metrics = compute_metrics(model, test_loader, criterion, device)

print(f"\n{'='*50}")
print(f"  TEST SET RESULTS")
print(f"{'='*50}")
print(f"  Total Loss:     {test_metrics['loss_total']:.6f}")
print(f"  Regression MSE: {test_metrics['loss_data']:.6f}")
print(f"  Class. BCE:     {test_metrics['loss_cls']:.6f}")
print(f"  KCL Penalty:    {test_metrics['loss_kcl']:.6f}")
print(f"  Thermo Penalty: {test_metrics['loss_thermo']:.6f}")
print(f"  -----")
print(f"  MAE:            {test_metrics['mae']:.6f} A/m2")
print(f"  RMSE:           {test_metrics['rmse']:.6f} A/m2")
print(f"  Accuracy:       {test_metrics['accuracy']:.4f}")
print(f"  F1 Score:       {test_metrics['f1']:.4f}")
print(f"{'='*50}")

## 🔍 Step 9: Prediction Analysis

In [ ]:
# Collect all test predictions
model.eval()
all_true_reg, all_pred_reg = [], []
all_true_cls, all_pred_cls = [], []

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        v_nodes, i_edges, compat = model(batch)

        i_true = batch.y_regression
        i_pred = i_edges[0::2].squeeze(-1)
        if i_pred.shape[0] != i_true.shape[0]:
            i_pred = i_edges.view(batch.num_graphs, 2, 1).mean(dim=1).squeeze(-1)

        all_true_reg.extend(i_true.cpu().numpy())
        all_pred_reg.extend(i_pred.cpu().numpy())
        all_true_cls.extend(batch.y_classification.cpu().numpy())
        all_pred_cls.extend(torch.sigmoid(compat).squeeze(-1).cpu().numpy())

all_true_reg = np.array(all_true_reg)
all_pred_reg = np.array(all_pred_reg)
all_true_cls = np.array(all_true_cls)
all_pred_cls = np.array(all_pred_cls)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Test Set - Prediction Analysis", fontsize=14, fontweight='bold')

axes[0].scatter(all_true_reg, all_pred_reg, alpha=0.2, s=8, c="steelblue")
lims = [min(all_true_reg.min(), all_pred_reg.min()), max(all_true_reg.max(), all_pred_reg.max())]
axes[0].plot(lims, lims, 'r--', linewidth=1.5, label="Perfect")
axes[0].set_xlabel("True Current Density (A/m2)")
axes[0].set_ylabel("Predicted Current Density (A/m2)")
axes[0].set_title("Regression: Predicted vs True")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

residuals = all_pred_reg - all_true_reg
axes[1].hist(residuals, bins=80, color="coral", edgecolor="white", alpha=0.8)
axes[1].axvline(x=0, color='k', linestyle='--', linewidth=1)
axes[1].set_xlabel("Residual (Predicted - True) A/m2")
axes[1].set_ylabel("Count")
axes[1].set_title(f"Residual Distribution (mean={residuals.mean():.4f})")
axes[1].grid(True, alpha=0.3)

compat_mask = all_true_cls == 0
unfav_mask = all_true_cls == 1
axes[2].hist(all_pred_cls[compat_mask], bins=50, alpha=0.6, label="Compatible (true)", color="green")
axes[2].hist(all_pred_cls[unfav_mask], bins=50, alpha=0.6, label="Unfavorable (true)", color="red")
axes[2].axvline(x=0.5, color='k', linestyle='--', linewidth=1, label="Threshold")
axes[2].set_xlabel("Predicted P(Unfavorable)")
axes[2].set_ylabel("Count")
axes[2].set_title("Classification Probability Distribution")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(P("prediction_analysis.png"), dpi=150, bbox_inches='tight')
plt.show()
print("Saved prediction_analysis.png")

## 🔮 Step 10: Predict on New Material Pairs

Use the trained model to predict corrosion behavior for specific material combinations.

In [ ]:
from graph_dataset import row_to_graph, _load_galvanic_lookup

galvanic_lookup = _load_galvanic_lookup(GALV_CSV)

def predict_pair(anode_name, cathode_name, environment, area_ratio=1.0):
    """Predict galvanic corrosion for a specific material pair."""
    anode_info = galvanic_lookup.get(anode_name, {})
    cathode_info = galvanic_lookup.get(cathode_name, {})

    v_anode = anode_info.get("potential_sce", 0)
    v_cathode = cathode_info.get("potential_sce", 0)
    delta_v = abs(v_cathode - v_anode)

    env_conductivity = {"Marine (Seawater)": 5.0, "Industrial (Acidic Rain)": 0.1, "Rural (Freshwater)": 0.01}
    row = pd.Series({
        "Anode_Alloy": anode_name,
        "Cathode_Alloy": cathode_name,
        "Anode_Table1_Group": anode_info.get("group", ""),
        "Cathode_Table1_Group": cathode_info.get("group", ""),
        "Potential_Difference_V": delta_v,
        "Environment": environment,
        "Electrolyte_Conductivity_Sm": env_conductivity.get(environment, 0.01),
        "Cathode_Anode_Area_Ratio": area_ratio,
        "Target_Galvanic_Current_Density": 0.0,
        "PDF_Compatibility_Status": "Compatible",
    })

    graph = row_to_graph(row, galvanic_lookup)

    from torch_geometric.data import Batch
    batch = Batch.from_data_list([graph]).to(device)

    model.eval()
    with torch.no_grad():
        v_nodes, i_edges, compat = model(batch)

    current_density = i_edges[0].item()
    prob_unfavorable = torch.sigmoid(compat).item()
    compatibility = "Unfavorable" if prob_unfavorable >= 0.5 else "Compatible"

    return {
        "anode": anode_name,
        "cathode": cathode_name,
        "environment": environment,
        "area_ratio": area_ratio,
        "delta_v": delta_v,
        "predicted_current_density": current_density,
        "compatibility": compatibility,
        "p_unfavorable": prob_unfavorable,
    }


test_pairs = [
    ("Zinc", "Copper", "Marine (Seawater)", 1.0),
    ("Magnesium and magnesium alloys", "Steel or iron", "Marine (Seawater)", 1.0),
    ("Commercially pure aluminum (1100)", "18-8 SS (passive)", "Industrial (Acidic Rain)", 2.0),
    ("Tin", "Lead", "Rural (Freshwater)", 1.0),
    ("Nickel (passive)", "Titanium", "Marine (Seawater)", 1.0),
    ("Zinc", "Platinum", "Marine (Seawater)", 10.0),
]

print(f"{'='*90}")
print(f"  INFERENCE - Material Pair Predictions")
print(f"{'='*90}")

results = []
for anode, cathode, env, ratio in test_pairs:
    r = predict_pair(anode, cathode, env, ratio)
    results.append(r)
    print(f"\n  {r['anode']:40s} <-> {r['cathode']}")
    print(f"    Environment: {r['environment']}, Area ratio: {r['area_ratio']}")
    print(f"    dV = {r['delta_v']:.3f} V  ->  i = {r['predicted_current_density']:.6f} A/m2")
    print(f"    Compatibility: {r['compatibility']}  (P(unfav) = {r['p_unfavorable']:.3f})")

df_results = pd.DataFrame(results)
print("\n")
display(df_results[["anode", "cathode", "environment", "delta_v",
                    "predicted_current_density", "compatibility", "p_unfavorable"]])

## 💾 Step 11: Download Results

In [ ]:
output_files = [
    P("checkpoints/best_model.pt"),
    P("checkpoints/final_model.pt"),
    P("training_log.csv"),
    P("training_curves.png"),
    P("prediction_analysis.png"),
    P("data_exploration.png"),
]

print("Output files:")
for f in output_files:
    if os.path.exists(f):
        size = os.path.getsize(f)
        print(f"  + {os.path.basename(f):40s}  ({size:,} bytes)")
    else:
        print(f"  - {os.path.basename(f):40s}  (not found)")

try:
    from google.colab import files
    print("\nDownloading files...")
    for f in output_files:
        if os.path.exists(f):
            files.download(f)
    print("Download complete.")
except ImportError:
    print("\n(Not running on Colab - files saved to disk)")

---

## Summary

| Component | Details |
|-----------|--------|
| **Architecture** | NNConv GNN (edge-aware) with 3 message passing layers |
| **Heads** | Node (potential), Edge (current density), Graph (compatibility) |
| **Physics Loss** | MSE + BCE + KCL conservation + Thermodynamic consistency |
| **Data** | 25,000 synthetic galvanic joints from 36 materials |
| **Source** | AMMTIAC compatibility tables (MIL-STD-889) |